# **UFO Sighting Analysis**

**By:** Macey Atkins and Saanvi Nair

**Project Site:** https://matkins8943.github.io/

# **Project Goals**
We are considering working on datasets related to paranormal UFO sightings/encounters.
The main question we are hoping to answer is whether or not we can predict the time and/or place of future UFO sightings based on previous data statistics and datasets.
We have currently found three datasets related to UFO sightings and object reports. We plan to use this data to aid our continued research.

# **Collaboration Plan**
We plan to continue using text messaging as the primary form of communication. We have created a private Github repository and plan to work on the project via phone calls and in person meetings once a week. Specific meeting dates will be dependent upon weekly availability.

# **Dataset 1**

Our first dataset was derived from [Kaggle](https://www.kaggle.com/datasets/ahemateja19bec1025/ufosightingsdataset). It measures UFO sightings across the years with the data, name, city/state/country, and description of the sighting. This set allows us to visualize the number of sightings over the United States per state. We want to know where most sightings occur and whether or not the region of the U.S. plays a role in reported sightings.


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import io
import plotly.express as px
from google.colab import drive
drive.mount('/content/drive')

#reads the file, replaces "None" values with NaN
df1 = pd.read_csv('/content/drive/MyDrive/UFO_Sightings.csv')
df1 = df1.replace("None", np.nan)

#creates the a dataframe of just observations (sightings) that occurred in the U.S.
unitedStates = df1[df1["Country"] == "United States"].copy()
#split "City, State" into 2 columns "City" and "State" to improve tidy data
unitedStates[['City', 'State']] = (unitedStates['City, State'].str.split(',', n=1, expand=True))

#removes irrelevant columns from the dataframe
unitedStates = unitedStates[['Date', 'Name', 'City', 'State', 'Country', 'Description']].copy()
#if only the "State" is listed (no city), then fill the "State" column with whatever is in the "City" column
unitedStates["State"] = unitedStates["State"].fillna(unitedStates["City"])
display(unitedStates.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Date,Name,City,State,Country,Description
11,1896–1897,Mystery airships,NaN,NaN,United States,Numerous reports of UFO sightings and attempte...
12,1897-04-17,"Aurora, Texas, UFO incident",Aurora,Texas,United States,A tale of a UFO crash and a burial of its alie...
16,1941,Cape Girardeau UFO crash,Cape Girardeau,Missouri,United States,First responders and a Baptist minister allege...
17,24-02-1942,Battle of Los Angeles,Los Angeles,California,United States,Unidentified aerial objects trigger the firing...
20,21-06-1947,Maury Island incident,Puget Sound near Maury Island,Washington,United States,Harold A. Dahl reported that his dog was kille...


Now that the dataset is in a cleaner format and zeroed in on occurrences in the U.S., we are able to create a heatmap to visualize the data. States with more reported sightings are represented in darker shades of blue, while states with fewer sightings are lighter.

In [8]:
#convert states to their corresponding abbreviations
stateabb = ({
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT",
    "Delaware": "DE", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND",
    "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA",
    "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "Virginia": "VA"
})

#some "States" are listed with other words (ex: "mostly Washington"), in order
#to decipher the state within, we use a for loop function to search if a
#state is in the "State" string
def find_state(key):
  if pd.isna(key):
    return np.nan
  for state, abbreviation in stateabb.items():
    if state in key:
      return abbreviation
  return np.nan

#strip states and replace them with their abbreviations
states = unitedStates['State'].str.strip()
newstateabb = states.apply(find_state)

#number of sightings (how many times each state has a reported sighting)
sightings = (newstateabb.value_counts().rename_axis('State').reset_index(name='Sightings'))

#creates the U.S. map
fig = px.choropleth(
    sightings,
    locations='State',
    locationmode='USA-states',
    color='Sightings',
    hover_name="State",
    color_continuous_scale='Blues',
    scope='usa',
    title="Sightings per U.S. State"
)

#layout of U.S. map figure
fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type='albers usa'),
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

As we can see, Texas and New Mexico have the highest number of reported sightings. The West Coast and Southern Boarder seem to have the most consistent set of sightings.

# **Dataset 2**

We found our second dataset from a [public repository](https://github.com/syamkakarla98/DataScience_Head_Start/blob/master/DataAnalysis/UFO_Sightings.ipynb) on github, free for public use. It also measures UFO sightings across the United States, but with higher specificity, listing the shapes that were identified from the sightings. We were curious about whether the geographic location of the city played a role in the shape that was seen.

In [ ]:
!pip install geonamescache -q
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import io
import plotly.express as px
import geonamescache
import plotly.express as px

df2 = pd.read_csv('https://bit.ly/uforeports')
df2 = df2.replace("None", np.nan)

#preparing it to be mapped
gc = geonamescache.GeonamesCache()
cities = pd.DataFrame(gc.get_cities().values())
print(cities.columns.tolist())
us = cities[cities["countrycode"] == "US"][["name", "admin1code", "latitude", "longitude", "population"]]
us.columns = ["City", "State", "latitude", "longitude", "population"]
us = us.sort_values("population", ascending=False).drop_duplicates(["City", "State"])

#matching lowercase
df2["key"] = df2["City"].str.lower().str.strip() + "|" + df2["State"]
us["key"] = us["City"].str.lower() + "|" + us["State"]

#mapping
mapped = df2.merge(us[["key", "latitude", "longitude"]], on="key", how="left")
print(f"matched {mapped['latitude'].notna().sum()} of {len(mapped)} rows")

plot_df = mapped.dropna(subset=["latitude", "Shape Reported"])

fig = px.scatter_geo(
    plot_df,
    lat="latitude", lon="longitude",
    color="Shape Reported",
    hover_name="City",
    scope="usa",
    title="UFO reports by shape"
)
fig.update_traces(marker=dict(size=5, opacity=0.6))
fig.show()





['geonameid', 'name', 'latitude', 'longitude', 'countrycode', 'population', 'timezone', 'admin1code', 'alternatenames']
matched 9956 of 18241 rows


# **Dataset 3**

In [ ]:
df3 = pd.read_csv("title.csv")
display(df3)